# Avaliação de Fusão: Classificador de Raças + Verificador de Identidade

Este experimento carrega dois modelos independentes:
1. **Classificador de Raças:** ConvNeXt-Tiny (treinado no Stanford Dogs)
2. **Verificador de Face:** ConvNeXt-Small + ArcFace (treinado no PetFace)

O objetivo é combinar as saídas de ambos os modelos para investigar se a inclusão da predição da raça consegue mitigar os Falsos Positivos de "cães diferentes, da mesma raça" que o verificador comete.

In [ ]:
# !pip install -q timm scikit-learn pandas matplotlib seaborn
import os
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
import timm

from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, f1_score

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## 1. Setup do Dataset e Caminhos
Assuma que você possui `classification_exp/results/best_model_stanford.pth`, `classification_exp/results/label_encoder_stanford.json`, e `dogs/best_arcface.pth` no seu Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive'
WORK_DIR = '/content/dogs'
os.makedirs(WORK_DIR, exist_ok=True)

# Defina caminhos dos modelos (ajuste conforme seu drive)
CLASSIFIER_WEIGHTS = f"{DRIVE_BASE}/classification_exp/results/best_model_stanford.pth"
LABEL_ENCODER_PATH = f"{DRIVE_BASE}/classification_exp/results/label_encoder_stanford.json"
VERIFIER_WEIGHTS = f"{DRIVE_BASE}/dogs/best_arcface.pth"
# Ajuste para utilizar o CSV original gravado no drive que contém 'positive', 'negative_same_breed', etc.
VERIF_CSV = f"{DRIVE_BASE}/dogs/split/verification_test_pairs.csv"
DATASET_DIR = f"{WORK_DIR}" # O CSV já contém o prefixo 'dog/' nos nomes dos arquivos

print(f"Buscando csv em: {VERIF_CSV}")
df_verif = pd.read_csv(VERIF_CSV)
print(f"Pares de verificação carregados: {len(df_verif)}")
if 'pair_type' in df_verif.columns:
    print("\nDistribuição de Pair Types:")
    print(df_verif['pair_type'].value_counts())

In [ ]:
import os

# Extrair dataset na máquina local do Colab (muito mais rápido que ler do Drive)
if not os.path.exists(DATASET_DIR):
    print("Extraindo dataset...")
    archive_candidates = [
        os.path.join(DRIVE_BASE, 'dogs', 'dog.zip'),
        os.path.join(DRIVE_BASE, 'dogs', 'dog.tar.gz'),
    ]
    archive_path = next((p for p in archive_candidates if os.path.exists(p)), None)

    if archive_path is None:
        print("⚠️ Nenhum arquivo compactado do dataset foi encontrado no Drive.")
        print(f"   Certifique-se de que {archive_candidates[0]} exista.")
    else:
        if archive_path.endswith('.zip'):
            !unzip -q "{archive_path}" -d "{WORK_DIR}"
        else:
            !tar -xzf "{archive_path}" -C "{WORK_DIR}"

print(f"Imagens prontas em: {DATASET_DIR}")

## 2. Definição das Arquiteturas

In [ ]:
# ---------------------------------------------------------
# Verificador de Identidade
# ---------------------------------------------------------
class EmbeddingModel(nn.Module):
    def __init__(self, embedding_dim=512, pretrained=False):
        super().__init__()
        self.backbone = timm.create_model('convnext_small.fb_in22k_ft_in1k', pretrained=pretrained, num_classes=0)
        self.embedding = nn.Sequential(
            nn.Linear(self.backbone.num_features, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
        )

    def forward(self, x):
        features = self.backbone(x)
        embeddings = self.embedding(features)
        return F.normalize(embeddings, p=2, dim=1)

# ---------------------------------------------------------
# Classificador de Raças
# ---------------------------------------------------------
def create_classifier(num_classes):
    # Usar torchvision em vez de timm para manter compatibilidade com o state_dict salvo
    model = models.convnext_tiny(weights=None)
    in_features = model.classifier[2].in_features
    model.classifier[2] = nn.Linear(in_features, num_classes)
    return model

In [ ]:
# Carregando os Modelos
print("Instanciando modelos...")

# Label Encoder para o classificador
with open(LABEL_ENCODER_PATH, 'r') as f:
    label_encoder = json.load(f)
num_classes = label_encoder['num_classes']

# Classificador
classifier = create_classifier(num_classes)
checkpoint_clf = torch.load(CLASSIFIER_WEIGHTS, map_location=DEVICE, weights_only=False)

# Caso o modelo tenha sido salvo com _orig_mod. ou module. (Dataparallel/Torch Compile)
state_dict_clf = checkpoint_clf.get('model_state_dict', checkpoint_clf)
new_state_dict = {}
for k, v in state_dict_clf.items():
    k = k.replace('_orig_mod.', '').replace('module.', '')
    new_state_dict[k] = v

classifier.load_state_dict(new_state_dict)
classifier.to(DEVICE)
classifier.eval()
print(f"Classificador carregado ({num_classes} classes).")

# Verificador
verifier = EmbeddingModel(embedding_dim=512, pretrained=False)
checkpoint_verif = torch.load(VERIFIER_WEIGHTS, map_location=DEVICE, weights_only=False)
verifier.load_state_dict(checkpoint_verif['backbone'])
verifier.to(DEVICE)
verifier.eval()
print("Verificador ArcFace (backbone) carregado.")

## 3. Lógica de Inferência

In [ ]:
eval_transform = T.Compose([
    T.Resize((224, 224), interpolation=T.InterpolationMode.BILINEAR),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class FusionDataset(Dataset):
    def __init__(self, df, root_dir, transform):
        self.df = df
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        p1 = os.path.join(self.root_dir, row['filename1'])
        p2 = os.path.join(self.root_dir, row['filename2'])

        img1 = Image.open(p1).convert('RGB')
        img2 = Image.open(p2).convert('RGB')

        if self.transform:
            t1 = self.transform(img1)
            t2 = self.transform(img2)

        label = int(row['label'])
        label = int(row['label'])

        # Determine pair_type if not present
        if 'pair_type' in row:
            pair_type = row['pair_type']
        else:
            id1 = row['filename1'].split('/')[1] if '/' in row['filename1'] else row['filename1']
            id2 = row['filename2'].split('/')[1] if '/' in row['filename2'] else row['filename2']
            if label == 1:
                pair_type = 'positive'
            else:
                pair_type = 'negative'

        return t1, t2, label, pair_type, p1, p2

test_dataset = FusionDataset(df_verif, DATASET_DIR, eval_transform)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)

In [ ]:
# Extrair Scores
all_labels = []
all_pair_types = []

sims_verifier = []
sims_breed_overlap = []

print("Executando inferência dupla...")
with torch.no_grad():
    for img1, img2, lbl, ptype, p1, p2 in tqdm(test_loader):
        img1, img2 = img1.to(DEVICE), img2.to(DEVICE)

        all_labels.extend(lbl.numpy())

        with torch.cuda.amp.autocast(dtype=torch.float16):
            # 1. Verificador -> Embeddings (512-d)
            emb1 = verifier(img1)
            emb2 = verifier(img2)

            # Cosine similarity crua
            cos_sim_verif = F.cosine_similarity(emb1, emb2, dim=1)
            # Re-escalar opcional de [-1, 1] para [0, 1] para facilitar fusão com probabilidades
            norm_sim_verif = (cos_sim_verif + 1.0) / 2.0

            sims_verifier.extend(norm_sim_verif.cpu().numpy())

            # 2. Classificador -> Logits
            logits1 = classifier(img1)
            logits2 = classifier(img2)

            probs1 = torch.softmax(logits1, dim=1)
            probs2 = torch.softmax(logits2, dim=1)

            # Produto interno das probabilidades das raças (O overlap da confiança)
            breed_overlap = torch.sum(probs1 * probs2, dim=1)
            sims_breed_overlap.extend(breed_overlap.cpu().numpy())

            # 3. Dynamic pair_type resolution for negatives if dataset only has generic 'negative'
            top1_1 = probs1.argmax(dim=1)
            top1_2 = probs2.argmax(dim=1)

            for i in range(len(lbl)):
                if ptype[i] and 'negative' in ptype[i] and ptype[i] not in ['negative_same_breed', 'negative_random', 'negative_diff_breed']:
                    if top1_1[i] == top1_2[i]:
                        all_pair_types.append('negative_same_breed')
                    else:
                        all_pair_types.append('negative_diff_breed')
                else:
                    all_pair_types.append(ptype[i])

all_labels = np.array(all_labels)
sims_verifier = np.array(sims_verifier)
sims_breed_overlap = np.array(sims_breed_overlap)

print("Extração concluída!")

## 4. Estratégias de Fusão e Avaliação TCC

In [ ]:
def evaluate_scores(scores, labels, pair_types, name="Strategy"):
    auc = roc_auc_score(labels, scores)
    fpr, tpr, thresholds = roc_curve(labels, scores)
    j_scores = tpr - fpr
    best_idx = np.argmax(j_scores)
    best_thresh = thresholds[best_idx]

    preds = (scores >= best_thresh).astype(int)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)

    # Análise de quebra
    df_eval = pd.DataFrame({'score': scores, 'label': labels, 'pred': preds, 'pair_type': pair_types})

    # Accuracy para o tipo mais difícil ('negative_same_breed')
    mask_hard = df_eval['pair_type'] == 'negative_same_breed'
    acc_hard_neg = accuracy_score(df_eval[mask_hard]['label'], df_eval[mask_hard]['pred']) if mask_hard.sum() > 0 else 0

    print(f"--- {name} ---")
    print(f"AUC: {auc:.4f} | Acc: {acc:.4f} | F1: {f1:.4f} | Thresh: {best_thresh:.4f}")
    if mask_hard.sum() > 0:
        print(f"Acc em Hard Negatives (same breed): {acc_hard_neg:.4f} ({mask_hard.sum()} pares)")
    print()

    return {'auc': auc, 'acc': acc, 'f1': f1, 'thresh': best_thresh, 'fpr': fpr, 'tpr': tpr}

# 1. Apenas Verificador (Baseline)
res_verif = evaluate_scores(sims_verifier, all_labels, all_pair_types, "Verificador Solo (ConvNeXt-S ArcFace)")

# 2. Fusão Multiplicativa
fuse_mult = sims_verifier * sims_breed_overlap
res_mult = evaluate_scores(fuse_mult, all_labels, all_pair_types, "Fusão: Multiplicativa (Verif * Breed)")

# 3. Fusão Linear (Alpha parameter sweep)
best_linear = None
best_auc_linear = 0
alphas = [0.1, 0.3, 0.5, 0.7, 0.8, 0.9]
print("--- Grid Search Linear (alpha * Verif + (1-alpha) * Breed) ---")
for a in alphas:
    fuse_lin = a * sims_verifier + (1 - a) * sims_breed_overlap
    auc = roc_auc_score(all_labels, fuse_lin)
    print(f"Alpha {a:.1f} -> AUC: {auc:.4f}")
    if auc > best_auc_linear:
        best_auc_linear = auc
        best_linear = (a, fuse_lin)

print()
res_linear = evaluate_scores(best_linear[1], all_labels, all_pair_types, f"Fusão: Linear Ótima (Alpha={best_linear[0]})")

# 4. Fusão Condicional / Bypass (Se breed overlap for minúsculo, rejeita direto)
# Isso foca em salvar falsos positivos de raças muito diferentes
thresh_breed = 0.05
fuse_bypass = np.where(sims_breed_overlap < thresh_breed, 0.0, sims_verifier)
res_bypass = evaluate_scores(fuse_bypass, all_labels, all_pair_types, f"Fusão: Bypass (Breed < {thresh_breed})")

In [ ]:
# Curvas ROC Comparativas
plt.figure(figsize=(8,6))

plt.plot(res_verif['fpr'], res_verif['tpr'], label=f"Verificador Solo (AUC = {res_verif['auc']:.4f})", linewidth=2)
plt.plot(res_linear['fpr'], res_linear['tpr'], label=f"Fusão Linear a={best_linear[0]} (AUC = {res_linear['auc']:.4f})", linewidth=2)
plt.plot(res_mult['fpr'], res_mult['tpr'], label=f"Fusão Multiplicativa (AUC = {res_mult['auc']:.4f})", linewidth=2)
plt.plot(res_bypass['fpr'], res_bypass['tpr'], label=f"Bypass Overlap<0.05 (AUC = {res_bypass['auc']:.4f})", linewidth=2, linestyle='--')

plt.plot([0,1], [0,1], 'k--', alpha=0.5)
plt.xlabel("False Positive Rate", fontsize=12)
plt.ylabel("True Positive Rate", fontsize=12)
plt.title("Comparação Curva ROC - Estratégias de Fusão", fontsize=14, fontweight='bold')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.savefig(os.path.join(WORK_DIR, "roc_fusion_comparison.png"), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Distribuição de Similaridade da Melhor Estratégia
best_strategy_scores = best_linear[1]
best_thresh = res_linear['thresh']

pos_mask = all_labels == 1
neg_mask = all_labels == 0

plt.figure(figsize=(10,5))
plt.hist(best_strategy_scores[neg_mask], bins=50, alpha=0.7, label='Negativo', color='#55A868', density=True)
plt.hist(best_strategy_scores[pos_mask], bins=50, alpha=0.7, label='Positivo', color='#ECBF6A', density=True)
plt.axvline(best_thresh, color='red', linestyle='--', label=f'Best Threshold = {best_thresh:.3f}')
plt.title(f"Distribuição de Scores na Fusão Linear (Alpha={best_linear[0]})", fontweight='bold')
plt.xlabel("Score Ponderado de Similaridade")
plt.ylabel("Densidade")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig(os.path.join(WORK_DIR, "score_distribution_fusion.png"), dpi=150, bbox_inches='tight')
plt.show()

## 5. Análise de Casos Recuperados
Buscando exemplos que o **Verificador Errou**, mas a **Fusão Acertou**.

In [ ]:
# Predições Baseline vs Fusão
preds_verif = (sims_verifier >= res_verif['thresh']).astype(int)
preds_fusion = (best_strategy_scores >= res_linear['thresh']).astype(int)

# Falso Positivo no verif cru, mas que virou True Negative na fusão
mask_recovered = (preds_verif == 1) & (all_labels == 0) & (preds_fusion == 0)

idx_recovered = np.where(mask_recovered)[0]
print(f"Falsos Positivos do Verificador que foram SALVOS (corrigidos) pela Fusão: {len(idx_recovered)}")

# Mostrar até 3 exemplos
num_show = min(3, len(idx_recovered))
import random

if num_show > 0:
    samples = random.sample(list(idx_recovered), num_show)
    fig, axes = plt.subplots(num_show, 2, figsize=(8, 4 * num_show))
    if num_show == 1: axes = [axes]

    for i, idx in enumerate(samples):
        row = df_verif.iloc[idx]
        img1 = Image.open(os.path.join(DATASET_DIR, row['filename1']))
        img2 = Image.open(os.path.join(DATASET_DIR, row['filename2']))

        ax1, ax2 = axes[i][0], axes[i][1]
        ax1.imshow(img1)
        ax1.axis('off')

        ax2.imshow(img2)
        ax2.axis('off')

        score_v = sims_verifier[idx]
        score_f = best_strategy_scores[idx]
        olap = sims_breed_overlap[idx]

        axes[i][0].set_title(f"Par Negativo (Diferentes)\nPair: {row.get('pair_type','?')}", fontsize=11)
        axes[i][1].set_title(f"Verificador: FP (Score {score_v:.2f})\nBreed Overlap: {olap:.3f}\nFusão Linear: TN (Score {score_f:.2f})!",
                             color='green', fontweight='bold', fontsize=10)

    plt.tight_layout()
    plt.savefig(os.path.join(WORK_DIR, "recovered_by_fusion.png"), dpi=150, bbox_inches='tight')
    plt.show()